In [3]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

# Set seed for reproducibility
np.random.seed(42)
random.seed(42)

print("Generating mock dataset for Power BI Dashboard...")
print("=" * 60)

# Generate date range for 2 months
start_date = datetime(2024, 1, 1)
end_date = datetime(2024, 2, 29)
date_range = [start_date + timedelta(days=i) for i in range((end_date - start_date).days + 1)]

# Create user registration data (200 users)
users = []
user_id_counter = 1

for date in date_range:
    # Variable number of new users each day
    if date.weekday() < 5:  # Weekdays have more registrations
        new_users = np.random.randint(2, 6)
    else:  # Weekends
        new_users = np.random.randint(1, 3)
    
    for _ in range(new_users):
        user_id = f"USER{user_id_counter:03d}"
        user_segment = random.choices(['Early Adopter', 'Regular', 'Late Adopter'], 
                                      weights=[0.2, 0.6, 0.2])[0]
        country = random.choices(['US', 'UK', 'CA', 'AU', 'DE', 'IN', 'JP'], 
                                 weights=[0.4, 0.2, 0.1, 0.1, 0.1, 0.05, 0.05])[0]
        
        users.append({
            'user_id': user_id,
            'registration_date': date.date(),
            'user_segment': user_segment,
            'country': country
        })
        user_id_counter += 1

# Create DataFrame
users_df = pd.DataFrame(users)

# Generate bot activity data
bot_types = ['Customer Service', 'Personal Assistant', 'E-commerce', 
             'Technical Support', 'HR Assistant', 'Marketing Bot']
bot_statuses = ['Active', 'Inactive', 'Pending']

activities = []
bot_id_counter = 1
conversation_id_counter = 1

print(f"Generating activities for {len(users_df)} users...")

for _, user in users_df.iterrows():
    user_id = user['user_id']
    registration_date = user['registration_date']
    
    # Each user creates 1-4 bots
    num_bots = np.random.randint(1, 5)
    
    for _ in range(num_bots):
        bot_id = f"BOT{bot_id_counter:03d}"
        bot_id_counter += 1
        
        # Bot creation date
        creation_offset = np.random.randint(0, 10)
        bot_creation_date = registration_date + timedelta(days=creation_offset)
        
        bot_type = random.choice(bot_types)
        bot_status = random.choices(bot_statuses, weights=[0.75, 0.15, 0.1])[0]
        
        # Generate conversations for this bot (2-10 conversations)
        num_conversations = np.random.randint(2, 11)
        
        for _ in range(num_conversations):
            # Conversation date
            conv_offset = np.random.randint(0, 30)
            conv_date = bot_creation_date + timedelta(days=conv_offset)
            
            # Ensure conversation date is within range
            if conv_date > end_date.date():
                conv_date = bot_creation_date + timedelta(days=np.random.randint(0, 15))
            
            conversation_id = f"CONV{conversation_id_counter:05d}"
            conversation_id_counter += 1
            
            # Generate conversation metrics
            messages_count = np.random.randint(3, 20)
            conversation_duration = round(np.random.uniform(1.0, 10.0), 1)
            satisfaction_score = np.random.randint(1, 6)  # 1-5 scale
            
            activities.append({
                'date': conv_date,
                'user_id': user_id,
                'bot_id': bot_id,
                'bot_type': bot_type,
                'bot_status': bot_status,
                'conversation_id': conversation_id,
                'messages_count': messages_count,
                'conversation_duration_minutes': conversation_duration,
                'satisfaction_score': satisfaction_score,
                'bot_creation_date': bot_creation_date,
                'user_country': user['country'],
                'user_segment': user['user_segment']
            })

# Create activities DataFrame
activities_df = pd.DataFrame(activities)

# Create summary tables
# 1. Daily new users
daily_new_users = users_df.groupby('registration_date').size().reset_index(name='new_users')

# 2. Bot type distribution
bot_type_dist = activities_df.groupby('bot_type').agg({
    'bot_id': 'nunique',
    'conversation_id': 'count',
    'messages_count': 'sum'
}).reset_index()
bot_type_dist.columns = ['bot_type', 'total_bots', 'total_conversations', 'total_messages']

# 3. Daily metrics
daily_metrics = activities_df.groupby('date').agg({
    'user_id': 'nunique',
    'conversation_id': 'count',
    'messages_count': 'sum',
    'conversation_duration_minutes': 'mean',
    'satisfaction_score': 'mean'
}).reset_index()
daily_metrics.columns = ['date', 'daily_active_users', 'daily_conversations', 
                         'daily_messages', 'avg_duration', 'avg_satisfaction']

# Display dataset statistics
print("\n" + "=" * 60)
print("DATASET STATISTICS")
print("=" * 60)
print(f"Total Users: {len(users_df)}")
print(f"Total Activities: {len(activities_df)}")
print(f"Total Bots: {activities_df['bot_id'].nunique()}")
print(f"Total Conversations: {activities_df['conversation_id'].nunique()}")
print(f"Date Range: {start_date.date()} to {end_date.date()}")
print(f"Bot Types: {', '.join(bot_types)}")
print(f"Countries: US, UK, CA, AU, DE, IN, JP")

print("\nSample Data (first 5 rows):")
print(activities_df[['date', 'user_id', 'bot_type', 'messages_count', 
                     'conversation_duration_minutes', 'satisfaction_score']].head())

print("\n" + "=" * 60)
print("CREATING EXCEL FILE FOR POWER BI...")
print("=" * 60)

# Save to Excel
with pd.ExcelWriter('bot_analytics_200_dataset.xlsx', engine='openpyxl') as writer:
    # Main datasets
    activities_df.to_excel(writer, sheet_name='activity_data', index=False)
    users_df.to_excel(writer, sheet_name='user_registrations', index=False)
    
    # Summary tables (useful for Power BI)
    daily_metrics.to_excel(writer, sheet_name='daily_metrics', index=False)
    daily_new_users.to_excel(writer, sheet_name='daily_new_users', index=False)
    bot_type_dist.to_excel(writer, sheet_name='bot_type_distribution', index=False)
    
    # Create a quick insights sheet
    insights_data = {
        'Metric': [
            'Total Users', 
            'Total Conversations', 
            'Total Bots Created',
            'Average Messages per Conversation',
            'Average Conversation Duration (min)',
            'Average Satisfaction Score',
            'Most Popular Bot Type',
            'Most Active Country'
        ],
        'Value': [
            len(users_df),
            len(activities_df),
            activities_df['bot_id'].nunique(),
            round(activities_df['messages_count'].mean(), 1),
            round(activities_df['conversation_duration_minutes'].mean(), 1),
            round(activities_df['satisfaction_score'].mean(), 1),
            activities_df['bot_type'].mode()[0],
            activities_df['user_country'].mode()[0]
        ]
    }
    insights_df = pd.DataFrame(insights_data)
    insights_df.to_excel(writer, sheet_name='key_insights', index=False)

print("✅ Excel file 'bot_analytics_200_dataset.xlsx' created successfully!")
print("\nSheets included:")
print("1. activity_data - Main dataset with all activities")
print("2. user_registrations - User registration data")
print("3. daily_metrics - Aggregated daily performance")
print("4. daily_new_users - New user registrations by day")
print("5. bot_type_distribution - Bot type analytics")
print("6. key_insights - Key metrics at a glance")

print("\n" + "=" * 60)
print("POWER BI VISUALIZATION GUIDE")
print("=" * 60)
print("\n1. NEW USERS OVER TIME (Line Chart):")
print("   Data: daily_new_users sheet")
print("   X-Axis: registration_date")
print("   Y-Axis: new_users")
print("   Title: 'Daily New User Registrations'")
print()
print("2. BOTS CREATED BY TYPE (Bar Chart):")
print("   Data: bot_type_distribution sheet")
print("   X-Axis: bot_type")
print("   Y-Axis: total_bots")
print("   Title: 'Bots Created by Type'")
print()
print("3. CARD VISUALS:")
print("   Total Users: Count of user_id from user_registrations")
print("   Total Conversations: Count of conversation_id from activity_data")
print("   Active Bots: Count of bot_id where bot_status = 'Active'")
print("   Avg Satisfaction: Average of satisfaction_score")
print()
print("4. ADDITIONAL VISUALIZATIONS:")
print("   • Conversation Volume Over Time")
print("   • User Geographic Distribution")
print("   • Bot Type Performance Comparison")
print("   • User Satisfaction Trends")

print("\n" + "=" * 60)
print("POWER BI SETUP STEPS:")
print("=" * 60)
print("1. Open Power BI Desktop")
print("2. Click 'Get Data' → 'Excel'")
print("3. Select 'bot_analytics_200_dataset.xlsx'")
print("4. Check all sheets and click 'Load'")
print("5. Create relationships:")
print("   - activity_data.user_id ↔ user_registrations.user_id")
print("   - activity_data.date ↔ daily_metrics.date")
print("   - activity_data.bot_type ↔ bot_type_distribution.bot_type")
print("6. Start building your dashboard!")

Generating mock dataset for Power BI Dashboard...
Generating activities for 189 users...

DATASET STATISTICS
Total Users: 189
Total Activities: 2742
Total Bots: 439
Total Conversations: 2742
Date Range: 2024-01-01 to 2024-02-29
Bot Types: Customer Service, Personal Assistant, E-commerce, Technical Support, HR Assistant, Marketing Bot
Countries: US, UK, CA, AU, DE, IN, JP

Sample Data (first 5 rows):
         date  user_id    bot_type  messages_count  \
0  2024-01-15  USER001  E-commerce               9   
1  2024-02-03  USER001  E-commerce              16   
2  2024-02-01  USER001  E-commerce               4   
3  2024-01-13  USER001  E-commerce              14   
4  2024-01-20  USER001  E-commerce              19   

   conversation_duration_minutes  satisfaction_score  
0                            1.1                   1  
1                            8.3                   1  
2                            7.2                   4  
3                            5.5                   3